# Dataset Exploration - Driver Drowsiness Detection

Quick sanity checks before training:
1. Class balance in `data/train`
2. A grid of sample images per class
3. A live demo of EAR/MAR landmark extraction on one sample image

Run this **after** placing the dataset under `data/train/{open_eye,closed_eye}/` as described in the README.

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src import config
from src.data_loader import _scan_split

train_counts = {cls: len(paths) for cls, paths in _scan_split(config.TRAIN_DIR).items()}
print('Training image counts:', train_counts)

plt.bar(train_counts.keys(), train_counts.values(), color=['#dc2626', '#16a34a'])
plt.title('Class balance - data/train')
plt.ylabel('Number of images')
plt.show()

In [ ]:
from src.data_loader import _scan_class_folder

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for row, cls in enumerate(config.EYE_STATE_CLASSES):
    paths = _scan_class_folder(config.TRAIN_DIR, cls)[:6]
    for col, p in enumerate(paths):
        img = Image.open(p).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls)
    for col in range(len(paths), 6):
        axes[row, col].axis('off')
fig.suptitle('Sample images per class')
plt.tight_layout()
plt.show()

## EAR / MAR landmark demo

Runs the same `FaceLandmarkDetector` used by the real-time app on a single
webcam-style frame (any face image works). This is mainly useful to sanity
-check the MediaPipe backend is installed correctly before running the full
app.

In [ ]:
import cv2
from src.face_detection import FaceLandmarkDetector

# Point this at any single face photo you have handy to sanity-check detection.
sample_path = None  # e.g. r'..\data\train\open_eye\some_face_photo.jpg'

if sample_path:
    frame = cv2.imread(sample_path)
    detector = FaceLandmarkDetector(backend='auto')
    result = detector.process(frame)
    print('backend:', detector.backend, '| success:', result.success, '| EAR:', result.ear, '| MAR:', result.mar)
    detector.close()
else:
    print('Set sample_path to a face image to try the detector interactively.')

## Next steps

```bash
python src/train.py --model both
python src/evaluate.py --model both
streamlit run app.py
```